# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and visualizing the FAIR^2 dataset: "Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency", using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

*This dataset contains structured tabulations of clinical features, hormone levels, imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.*

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

We query the Croissant metadata for all available record sets, their fields, and columns. To maintain consistency, **all entities are referenced by their `@id` fields** according to the Croissant schema.

In [ ]:
# List all record sets and their details
record_sets = dataset.record_sets

print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"  RecordSet: {rs['@id']}, Name: {rs.get('name', 'NA')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"    Field: {field['@id']}, Name: {field.get('name', 'NA')}, DataType: {field.get('dataType', 'NA')}")

For a quick preview of the data, let's print a few example records from each available record set (using their `@id`).

In [ ]:
# Preview some records from each record set
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nSample records from RecordSet {rs_id}:")
    records = dataset.records(record_set=rs_id)
    for i, rec in enumerate(records):
        print(rec)
        if i >= 2:  # Show only first 3 records
            break

## 3. Data Extraction

Load data from each record set into pandas DataFrames for analysis. The record sets and fields are referenced using their `@id` values. This allows consistent programmatic access aligned with Croissant best practices.

We'll automatically extract the record set IDs and construct DataFrames for each.

In [ ]:
# Build DataFrames for each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"DataFrame for RecordSet {rs_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's process one record set in detail. We'll:
- Filter data by numeric field (e.g., 'age') using its `@id`.
- Normalize the numeric field.
- Group by a categorical attribute using its `@id` (if available).

> **Note:** Replace `<numeric_field_id>` and `<group_field_id>` with valid field IDs from the previous overview. This section demonstrates analysis using proper Croissant `@id` referencing.

In [ ]:
# Choose a RecordSet for analysis
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]

    # Find a numeric field (example: age at diagnosis). We'll search for 'age' in column names.
    numeric_field_id = None
    for col in df.columns:
        if 'age' in col.lower():
            numeric_field_id = col
            break

    if numeric_field_id:
        # Filter: age > threshold
        threshold = 30
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df)

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a categorical attribute (e.g., 'hirsutism', 'infertility', etc.)
        group_field_id = None
        for col in df.columns:
            if 'hirsutism' in col.lower():
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No suitable numeric field (e.g., age) found for EDA.")
else:
    print("No record sets available.")

## 5. Visualization

Visualize a distribution or relationship within the dataset using matplotlib. We'll create a histogram of the selected numeric field (e.g., 'age at diagnosis').

In [ ]:
import matplotlib.pyplot as plt

if record_set_ids and numeric_field_id:
    plt.figure(figsize=(6,4))
    df = dataframes[rs_id]
    plt.hist(df[numeric_field_id].dropna(), bins=5, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we explored the FAIR^2 clinical dataset using `mlcroissant`, referencing data entities by their Croissant `@id`. We loaded and previewed data, extracted DataFrames, performed basic EDA filtering and normalization, and visualized a numeric field.

- **Key findings:**
    - The dataset comprises structured clinical, laboratory, imaging, and genetic records for three female patients.
    - Croissant entity IDs provide consistent reference points for extracting, processing, and visualizing data.
    - The small sample size is suitable for case illustration and academic exploration but not predictive modeling.

Feel free to extend this notebook for further hypothesis exploration, additional visualization, or integration with other FAIR-compliant datasets.